<a href="https://colab.research.google.com/github/aWolander/google-colab/blob/main/Federated%20Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import random
import numpy as np
# vits16 = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')

# print(vits16)


In [ ]:

def main():
    K= 100
    N = 10
    CIFAR100 = datasets.CIFAR100(
      root="data",
      download=True,
      transform=ToTensor()
    )
    train_dataset, validate_dataset, test_dataset = torch.utils.data.random_split(CIFAR100, [0.7,0.15,0.15])
    client_datasets = split_data_iid(train_dataset, K)
    test_split(client_datasets)

def create_label_indexing(dataset):
    label_index = {i: [] for i in range(100)}
    for idx, (_, label) in enumerate(dataset):
        label_index[label].append(idx)
    return label_index

def split_data_non_iid(dataset, K, N_c = 100):
    index_split = [ [] for _ in range(K) ]
    label_index = create_label_indexing(dataset)
    split_availible_labels = {}
    lablels_exhausted = [False]*K

    # Randomly choose which labels each client has access to
    for i in range(K):
        split_availible_labels[i] = random.sample(range(100), N_c)

    # stops if no more datapoints are left to assign to any client
    while not all(lablels_exhausted):
        for client in range(K):

            if not split_availible_labels[client]:
                lablels_exhausted[client] = True

            for label in split_availible_labels[client]:
                # if no indices are left for a label, remove the label
                if not label_index[label]:
                    split_availible_labels[client].remove(label)
                    continue

                index_split[client].append(label_index[label].pop())

    for client_indices in index_split:
        random.shuffle(client_indices)

    return [Subset(dataset, indices) for indices in index_split]


def split(l: list, n: int) -> list[list]:
    '''splits a list into n parts'''
    split_list = []
    for i in range(0, n):
        split_list.append(l[i::n])
    return split_list

def split_data_iid(dataset, K):
    index_split = [[] for x in range(K)]
    label_index = create_label_indexing(dataset)

    for label in range(0,100):
        split_indices = split(label_index[label],K)
        for client in range(0, K):
            index_split[client] += split_indices[client]

    for client_indices in index_split:
        random.shuffle(client_indices)
    return [Subset(dataset, indices) for indices in index_split]

def test_split(client_datasets):
    bottom = np.zeros(100)
    for client_id, client_dataset in enumerate(client_datasets):
        occurrences = np.zeros(100)
        for datapoint in client_dataset:
            label = datapoint[1]
            occurrences[label] += 1
        #print(occurrences)
        print(f"Client {client_id}: No. of nonzero elements: {np.count_nonzero(occurrences)} | Avg., stdev. of nonzero elements: {occurrences[occurrences>0].mean()}, {occurrences[occurrences>0].std()}") # should be N_c non-zero elements!
        plt.bar(range(100), occurrences, bottom=bottom, label=client_id)
        plt.xlabel("Class label")
        plt.ylabel("Number of samples")
        bottom += occurrences

    plt.show()

main()

# def test_split_index(dataset):
#     K= 2
#     N = 5
#     client_data_split = split_data_non_iid(dataset, K, N)
#     for client_data in client_data_split:
#         occurences = {}
#         for index in client_data:
#             occurences[dataset[index][1]] = occurences.setdefault(dataset[index][1], 0) + 1
#         print(occurences)
#         plt.hist(occurences, stacked=True, bins=100)
#     plt.show()

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
class DinoFullModel(nn.Module):
    def __init__(self):
      super().__init__()
      # Load the full DINO model (backbone + head)
      self.model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16').to(device)

      self.learning_rate = 1e-2
      self.epochs = 10

      self.loss_fn = nn.CrossEntropyLoss()
      self.optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate)

      self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        self.optimizer, T_max=self.epochs
      )

    def forward(self, x):
      return self.model(x)

    def train_model(self, dataloader):
      size = len(dataloader.dataset)
      # Set the model to training mode - important for batch normalization and dropout layers
      # Unnecessary in this situation but added for best practices
      self.train()
      for epoch in range(self.epochs):
        current = 0
        print(f"-------------------------------\nEpoch {epoch+1}\n-------------------------------")
        for (X, y) in dataloader:
          # Compute prediction and loss
          X = X.to(device)
          y = y.to(device)
          pred = self.model(X).to(device)
          loss = self.loss_fn(pred, y)

          # Backpropagation
          loss.backward()
          self.optimizer.step()
          self.optimizer.zero_grad()
          current += len(X)
          if current % 5000 == 0:
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
            loss = loss.item()

        self.scheduler.step()



    def test_model(self, dataloader):
      self.model.eval()
      total_loss = 0.0
      total_correct = 0
      total_samples = 0

      with torch.no_grad():
        for inputs, targets in dataloader:
          inputs, targets = inputs.to(device), targets.to(device)

          outputs = self.model(inputs)
          loss = self.loss_fn(outputs, targets)
          total_loss += loss.item() * inputs.size(0)  # total loss, not average

          # Get predicted class
          preds = outputs.argmax(dim=1)
          total_correct += (preds == targets).sum().item()
          total_samples += targets.size(0)

      avg_loss = total_loss / total_samples
      accuracy = total_correct / total_samples

      #print(f"Validation Loss: {avg_loss:.4f}, Correct: {total_correct} out of {len(dataloader)}, Accuracy: {accuracy:.2%}")

      return avg_loss, total_correct, accuracy

def main():

  train_dataset, validate_dataset, test_dataset = torch.utils.data.random_split(CIFAR100, [0.7,0.15,0.15])

  train_dataloader = DataLoader(train_dataset, batch_size=100)
  test_dataloader = DataLoader(test_dataset, batch_size=100)
  validate_dataloader = DataLoader(validate_dataset, batch_size=100)

  model = DinoFullModel().to(device)
  #for param in model.parameters():
  #    print(param.shape)

  model.train_model(train_dataloader)
  model.test_model(test_dataloader)
  print("Done!")

if __name__=="__main__":
  main()

In [ ]:
'''
TODO:
Batch normalization, probably. I think I read somewhere that the facebook model needs this
Version control and checkpointing
Testing and hyperparameter tuning
gradient mask TaLoS thing. Also expanding this to FL.
Plot accuracy and loss method in DinoFullModel
'''

K=5
FL = FL_server(K)
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
CIFAR100 = datasets.CIFAR100(
  root="data",
  download=True,
  transform=ToTensor()
)
train_dataset, validate_dataset, test_dataset = torch.utils.data.random_split(CIFAR100, [0.7,0.15,0.15])


split_dataset = split_data_iid(train_dataset, K)

train_dataloader = [DataLoader(subset, batch_size=100) for subset in split_dataset]

test_dataloader = DataLoader(test_dataset, batch_size=100)
validate_dataloader = DataLoader(validate_dataset, batch_size=100)

FL.FedAvg(train_dataloader, validate_dataloader, 1)
FL.test_model(test_dataloader)

